# Recorded main compression workflow

This notebook reproduces the completed 21 October Kaggle compression stage.
Read `README.md` in the repository first. Existing final results need no new GPU run.

For an intentional reproduction, attach the preserved
`outputs/handoffs/symbiote_main_stage_c_v1.zip` as a Kaggle input. Restore the five
recorded checkpoints under `/kaggle/working/symbiote/models/span_scorer_v2_20261021/`
before setup; they are not included in the transfer. Enable GPU and Internet.
Run the four cells in order. Setup must pass before compression. The last cell
only exports saved artifacts and can run independently after a failure.

The transfer pins the original source bytes and manifest, preserves outputs,
and verifies checkpoints. It generates 2,240 compressed prompts, with no target
API calls or training. The two hosted-reader runs are documented in README.md
and RESULTS.md; they have already finished.


In [ ]:
from pathlib import Path
import hashlib, zipfile

inputs = Path("/kaggle/input")
expanded = sorted(inputs.rglob("kaggle_stage_c_bootstrap.py"))
archives = sorted(inputs.rglob("symbiote_main_stage_c_v1.zip"))
if expanded:
    source = expanded[0].parent.parent
    script = expanded[0].read_bytes()
elif archives:
    source = archives[0]
    with zipfile.ZipFile(source) as archive:
        script = archive.read("final_eval/kaggle_stage_c_bootstrap.py")
else:
    raise FileNotFoundError("Attach the NEW main-stage transfer ZIP as a Kaggle input dataset first.")
assert hashlib.sha256(script).hexdigest() == "2eac2b960d1e17a0c6e6747702fd898ab2fbc25e050959dc76e568dff4c9e871", "Wrong transfer version; use the matching cells"
namespace = {"__name__": "main_transfer_bootstrap"}
exec(compile(script, "verified_stage_c_bootstrap", "exec"), namespace)
namespace["install"](source, "c07669284025dd2c434ba02583069c11af5a081e1eec9b133b2b732df54ca322")


In [ ]:
from pathlib import Path
import subprocess, sys

root = Path("/kaggle/working/symbiote")
subprocess.run([sys.executable, "-u", "-m", "final_eval.kaggle_main", "setup"], cwd=root, check=True)


In [ ]:
from pathlib import Path
import subprocess, sys

root = Path("/kaggle/working/symbiote")
subprocess.run([sys.executable, "-u", "-m", "final_eval.kaggle_main", "run"], cwd=root, check=True)


In [ ]:
from pathlib import Path
import subprocess, sys
from IPython.display import display, FileLink

root = Path("/kaggle/working/symbiote")
# Safe to run alone after an error/interruption; this only exports saved files.
subprocess.run([sys.executable, "-u", "-m", "final_eval.kaggle_main", "export"], cwd=root, check=True)
archives = sorted(Path("/kaggle/working").glob("symbiote_main_review_*.zip"), key=lambda p: p.stat().st_mtime)
print("Download this ZIP and put it in the local project folder:", archives[-1])
display(FileLink(str(archives[-1])))
